# Simulation
Simulation is the process of using a computer to mimic a physical experiment. In this class, those experiments will almost invariably involve chance. 

We have seen how to simulate the results of tosses of a coin. The steps in that simulation were examples of the steps that will constitute every simulation we do in this course. In this section we will set out those steps and follow them in examples.

## The Process

**Step 1: What to Simulate**

Decide which quantity you want to simulate. For example, you might decide that you want to simulate the outcomes of tosses of a coin. Each simulated value will be a Head or a Tail.

**Step 2: Simulating One Value**

Figure out how to simulate *one* value of the quantity you specified in Step 1. In our example, you have to figure out how to simulate the outcome of *one* toss of a coin. If your quantity is more complicated, you might need several lines of code to come up with one simulated value. Typically, we will define a function that returns the simulated value.

**Step 3: Number of Repetitions**

Decide how many times you want to simulate the quantity. You will have to repeat the simulation in Step 2 that many times. In one of our earlier examples we had decided to simulate the outcomes of 1000 tosses of a coin, and so we needed 1000 repetitions of generating the outcome of a single toss.

**Step 4: Simulating Multiple Values**

Finally, put it all together as follows.

- Create an empty Series in which to collect all the simulated values. We will call this the collection Series.
- Create a "repetitions sequence," that is, a sequence whose length is the number of repetitions you specified in Step 3. For `n` repetitions we will almost always use the sequence `range(n)`.
- Create a `for` loop. For each element of the repetitions sequence:
    - Simulate *one* value by using the function you wrote in Step 2.
    - Augment the collection Series with this simulated value.
    
That's it! Once you have carried out the steps above, your simulation is done. The collection Series contains all the simulated values. 

At this point you can use the collection Series as you would any other Series. You can count how many simulated values fall into a particular category, place the Series in the column of a DataFrame and visualize the distribution of the simulated values, and so on.

In [1]:
import polars as pl
import hvplot.polars
path_data = '../../../assets/data/'


## Example: Number of Heads in 100 Tosses
It is natural to expect that in 100 tosses of a coin, there will be 50 heads, give or take a few. 

But how many is "a few"? What's the chance of getting exactly 50 heads? Questions like these matter in data science not only because they are about interesting aspects of randomness, but also because they can be used in analyzing experiments where assignments to treatment and control groups are decided by the toss of a coin.

In this example we will simulate the number of heads in 100 tosses of a coin. The histogram of our results will give us some insight into how many heads are likely.

Let's get started on the simulation, following the steps above.

**Step 1: What to Simulate**

The quantity we are going to simulate is the number of heads in 100 tosses. 

**Step 2: Simulating One Value**

We have to figure out how to make one set of 100 tosses and count the number of heads. Let's start by creating a coin.

In [2]:
coin = pl.Series("coin", ["Heads", "Tails"])


In our earlier example we used `sample` and a `for` loop to generate multiple tosses. But sets of coin tosses are needed so often in data science that `sample` simulates them for us if we include an argument that is the number of times to toss, along with `with_replacement=True`.

Here are the results of 10 tosses.

In [3]:
# POLARS_CHANGE: with_replacement=True matches np.random.choice, which draws with replacement
ten_tosses = coin.sample(10, with_replacement=True)
ten_tosses


coin
str
"""Tails"""
"""Heads"""
"""Heads"""
"""Tails"""
"""Tails"""
"""Tails"""
"""Heads"""
"""Tails"""
"""Heads"""


We can count the number of heads by summing the Boolean Series, as before:

In [4]:
(ten_tosses == "Heads").sum()


5

Our goal is to simulate the number of heads in 100 tosses, not 10. To do that we can just repeat the same code, replacing 10 by 100.

In [5]:
outcomes = coin.sample(100, with_replacement=True)
num_heads = (outcomes == "Heads").sum()
num_heads


53

Since we will want to do this multiple times, let's define a function that returns the simulated value of the number of heads. We can do this using the code developed in the cell above.

In [6]:
def one_simulated_value():
    outcomes = coin.sample(100, with_replacement=True)
    return (outcomes == "Heads").sum()


**Step 3: Number of Repetitions**

How many repetitions we will use is up to us. The more we use, the more reliable our simulations will be, but the longer it will take to run the code. Python is pretty fast at tossing coins, so let's go for 20,000 repetitions. That means we are going to do the following 20,000 times:
- Toss a coin 100 times and count the number of heads.

That's a lot of tossing! It's good that we have Python to do it for us.

**Step 4: Simulating Multiple Values**

We are ready to create a Series of 20,000 simulated values of the number of heads in 100 tosses of a coin.

In [7]:
num_repetitions = 20000   # number of repetitions

heads = pl.Series("heads", [], dtype=pl.Int64) # empty collection Series

for i in range(num_repetitions):   # repeat the process num_repetitions times
    new_value = one_simulated_value()  # simulate one value using the function defined
    heads = pl.concat([heads, pl.Series(heads.name, [new_value])]) # augment the collection Series with the simulated value

# That's it! The simulation is done.


Check that the Series `heads` contains 20,000 entries, one for each repetition of the experiment.

In [8]:
len(heads)


20000

To get a sense of the variability in the number of heads in 100 tosses, we can collect the results in a DataFrame and draw a histogram.

In [9]:
simulation_results = pl.DataFrame([
    pl.int_range(1, num_repetitions + 1, eager=True).alias("Repetition"),
    heads.alias("Number of Heads"),
])


In [10]:
simulation_results.head(3)


Repetition,Number of Heads
i64,i64
1,53
2,46
3,52


In [11]:
bins = pl.linear_space(30.5, 69.5, 40, closed="both", eager=True)
simulation_results.hvplot.hist(
    "Number of Heads",
    bins=bins.to_list(),
    normed=True,
    xlabel="Number of Heads",
)


:Histogram   [Number of Heads]   (Frequency)

Each bin has width 1 and is centered at each value of the number of heads.

Not surprisingly, the histogram looks roughly symmetric around 50 heads. The height of the bar at 50 is about 8% per unit. Since each bin is 1 unit wide, this is the same as saying that about 8% of the repetitions produced exactly 50 heads. That's not a huge percent, but it's the largest compared to the percent at every other number of heads.

The histogram also shows that in almost all of the repetitions, the number of heads in 100 tosses was somewhere between 35 and 65. Indeed, the bulk of the repetitions produced numbers of heads in the range 45 to 55. 

While in theory it is *possible* that the number of heads can be anywhere between 0 and 100, the simulation shows that the range of *probable* values is much smaller.

This is an instance of a more general phenomenon about the variability in coin tossing, as we will see later in the course.

## Example: Moves in Monopoly ##

Each move in the game Monopoly is determined by the total number of spots of two rolls of a die. If you play Monopoly, what should you expect to get when you roll the die two times?

We can explore this by simulating the sum of two rolls of a die. We will run the simulation 10,000 times. Notice that in this paragraph we have completed Steps 1 and 3 of our simulation process.

Step 2 is the one in which we write a function to simulate the total number of spots in one pair of rolls. First, let's plan our code. We will create a Series containing the numbers 1 through 6, draw two times from the Series at random with replacement, and add the two numbers drawn.

In [12]:
die = pl.int_range(1, 7, eager=True).alias("die")
sum(die.sample(2, with_replacement=True))


2

We can use the Series `die` and the expression above to define a function that simulates one move in Monopoly.

In [13]:
def one_simulated_move():
    return sum(die.sample(2, with_replacement=True))


Now we can create a Series of 10000 simulated Monopoly moves, by starting with an empty collection Series and augmenting it by each new simulated move.

In [14]:
num_repetitions = 10000

moves = pl.Series("moves", [], dtype=pl.Int64)
for i in range(num_repetitions):
    new_move = one_simulated_move()
    moves = pl.concat([moves, pl.Series(moves.name, [new_move])])


Here is a histogram of the results.

In [15]:
results = pl.DataFrame([
    pl.int_range(1, num_repetitions + 1, eager=True).alias("Repetition"),
    moves.alias("Sum of Two Rolls"),
])

bins = pl.linear_space(1.5, 12.5, 12, closed="both", eager=True)
results.hvplot.hist(
    "Sum of Two Rolls",
    bins=bins.to_list(),
    normed=True,
    xlabel="Sum of Two Rolls",
)


:Histogram   [Sum of Two Rolls]   (Frequency)

Seven is the most common value, with the frequencies falling off symmetrically on either side.